# Lab 4 - Structured Streaming on Wikimedia events

The producer (`wikistream_producer.py`) sends the Wikimedia `recentchange` events to the Kafka topic `wikistreams`.
This notebook reads the topic with Spark Structured Streaming and runs several streaming aggregations.

Every query writes to the **console sink**, so the results appear in the Docker logs:

```bash
docker logs -f pyspark_notebook
```

Each result table has a `query` column so we can tell the queries apart in the logs.

| # | Query | Window | Output mode |
|---|-------|--------|-------------|
| 0 | Demo: edits by bot / human | tumbling 1 hour | complete |
| 1 | Activity and edit size per minute | **tumbling** 1 min (watermark 2 min) | update |
| 2 | Most edited pages | **sliding** 10 min, every 2 min | complete |
| 3 | Most active human users | **sliding** 5 min, every 1 min | complete |
| 4 | Edit size categories | **tumbling** 2 min (watermark 2 min) | update |
| 5 | Activity per wiki (useful with several `--wiki`) | **tumbling** 1 min (watermark 2 min) | update |
| 6 | Big edits alerts (no aggregation) + parquet in `data/` | none | append |

In [1]:
# Imports cell
from pyspark.sql import SparkSession
from pyspark.sql.functions import (
    col, from_json, window, count, sum as sum_, avg, max as max_, min as min_,
    abs as abs_, when, lit, coalesce, timestamp_seconds, approx_count_distinct, round as round_
)
from pyspark.sql.types import StructType, StructField, StringType, LongType, IntegerType, BooleanType

In [3]:
# initialize the spark session
# The Kafka connector must have the same version as Spark (the image is updated often:
# a 4.1.0 connector fails on Spark 4.2.0 with a NoSuchMethodError)
import pyspark
kafka_package = f"org.apache.spark:spark-sql-kafka-0-10_2.13:{pyspark.__version__}"

spark = SparkSession.builder \
    .appName("KafkaStreamingPrototype") \
    .config("spark.jars.packages", kafka_package) \
    .config("spark.sql.shuffle.partitions", "4") \
    .config("spark.sql.session.timeZone", "UTC") \
    .getOrCreate()

# Less noise in the Docker logs
spark.sparkContext.setLogLevel("WARN")
spark.version

'4.2.0'

In [4]:
# Define the initial raw streaming dataframe
raw_df = spark \
    .readStream \
    .format("kafka") \
    .option("kafka.bootstrap.servers", "kafka:19092") \
    .option("subscribe", "wikistreams") \
    .option("startingOffsets", "earliest") \
    .option("failOnDataLoss", "false") \
    .load()
# failOnDataLoss=false: the topic is lost when the Kafka container is recreated,
# the parquet checkpoint in data/ then points to offsets that no longer exist

In [5]:
# Schema with the fields used in the analysis
wikimedia_schema = StructType([
    StructField("title", StringType(), True),
    StructField("user", StringType(), True),
    StructField("timestamp", LongType(), True),
    StructField("type", StringType(), True),
    StructField("wiki", StringType(), True),
    StructField("server_name", StringType(), True),
    StructField("namespace", IntegerType(), True),
    StructField("bot", BooleanType(), True),
    StructField("minor", BooleanType(), True),
    StructField("comment", StringType(), True),
    StructField("length", StructType([
        StructField("old", IntegerType(), True),
        StructField("new", IntegerType(), True)
    ]), True)
])

# Apply the schema to organize the raw input
parsed_df = raw_df.select(
    from_json(col("value").cast("string"), wikimedia_schema).alias("data")
).select("data.*")

# Event time (Unix epoch in seconds -> timestamp) and size of each edit in bytes
edits_df = parsed_df \
    .withColumn("event_time", timestamp_seconds(col("timestamp"))) \
    .withColumn("size_delta", coalesce(col("length.new"), lit(0)) - coalesce(col("length.old"), lit(0))) \
    .withColumn("abs_delta", abs_(col("size_delta")))

edits_df.printSchema()

root
 |-- title: string (nullable = true)
 |-- user: string (nullable = true)
 |-- timestamp: long (nullable = true)
 |-- type: string (nullable = true)
 |-- wiki: string (nullable = true)
 |-- server_name: string (nullable = true)
 |-- namespace: integer (nullable = true)
 |-- bot: boolean (nullable = true)
 |-- minor: boolean (nullable = true)
 |-- comment: string (nullable = true)
 |-- length: struct (nullable = true)
 |    |-- old: integer (nullable = true)
 |    |-- new: integer (nullable = true)
 |-- event_time: timestamp (nullable = true)
 |-- size_delta: integer (nullable = false)
 |-- abs_delta: integer (nullable = false)



In [6]:
# Helper: start a console query (output goes to the Docker logs)
def start_console(df, name, output_mode, trigger="30 seconds", num_rows=20):
    return df \
        .select(lit(name).alias("query"), "*") \
        .writeStream \
        .queryName(name) \
        .format("console") \
        .outputMode(output_mode) \
        .option("truncate", False) \
        .option("numRows", num_rows) \
        .trigger(processingTime=trigger) \
        .start()

## 0. Demo: number of edits by bots and humans (tumbling window of 1 hour)

In [7]:
# A simple aggregation of raw data by number of bot edits
windowed_counts_df = edits_df \
    .groupBy(
        window(col("event_time"), "1 hour"),
        col("bot")
    ) \
    .agg(count("*").alias("edit_count")) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        col("bot"),
        col("edit_count")
    )

q0 = start_console(windowed_counts_df, "q0_demo_bot_1h", "complete")

## 1. Tumbling window: activity and edit size per minute

Tumbling windows do not overlap: every edit belongs to exactly one 1-minute window.

The watermark of 2 minutes lets Spark drop the state of old windows. In `update` mode, only the windows that changed in the micro-batch are printed.

In [8]:
activity_1min_df = edits_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(window(col("event_time"), "1 minute")) \
    .agg(
        count("*").alias("edits"),
        sum_(when(col("bot"), 1).otherwise(0)).alias("bot_edits"),
        sum_(when(col("minor"), 1).otherwise(0)).alias("minor_edits"),
        approx_count_distinct("user").alias("users"),
        approx_count_distinct("title").alias("pages"),
        sum_(when(col("size_delta") > 0, col("size_delta")).otherwise(0)).alias("bytes_added"),
        sum_(when(col("size_delta") < 0, -col("size_delta")).otherwise(0)).alias("bytes_removed"),
        round_(avg("abs_delta"), 1).alias("avg_edit_size"),
        max_("abs_delta").alias("biggest_edit")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "edits", "bot_edits", "minor_edits", "users", "pages",
        "bytes_added", "bytes_removed", "avg_edit_size", "biggest_edit"
    )

q1 = start_console(activity_1min_df, "q1_tumbling_1min_activity", "update")

## 2. Overlapping (sliding) window: most edited pages

Windows of 10 minutes starting every 2 minutes: they overlap, so one edit is counted in 5 windows.
This gives a smoothed view of the "hot" pages.

Sorting is only allowed in `complete` mode. The most recent windows are shown first, then the pages with the most edits.

In [9]:
hot_pages_df = edits_df \
    .groupBy(
        window(col("event_time"), "10 minutes", "2 minutes"),
        col("title")
    ) \
    .agg(
        count("*").alias("edits"),
        approx_count_distinct("user").alias("users"),
        sum_("size_delta").alias("net_bytes")
    ) \
    .filter(col("edits") >= 2) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "title", "edits", "users", "net_bytes"
    ) \
    .orderBy(col("window_start").desc(), col("edits").desc())

q2 = start_console(hot_pages_df, "q2_sliding_10min_hot_pages", "complete")

## 3. Overlapping (sliding) window: most active human users

Windows of 5 minutes sliding every minute, bots removed.

In [10]:
active_users_df = edits_df \
    .filter(~col("bot")) \
    .groupBy(
        window(col("event_time"), "5 minutes", "1 minute"),
        col("user")
    ) \
    .agg(
        count("*").alias("edits"),
        approx_count_distinct("title").alias("pages"),
        sum_("abs_delta").alias("bytes_changed")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "user", "edits", "pages", "bytes_changed"
    ) \
    .orderBy(col("window_start").desc(), col("edits").desc())

q3 = start_console(active_users_df, "q3_sliding_5min_active_users", "complete")

## 4. Tumbling window: distribution of the edit sizes

Each edit is put in a size category, then counted per 2-minute window.

In [11]:
size_category = when(col("size_delta") < 0, "1. removal") \
    .when(col("size_delta") == 0, "2. no size change") \
    .when(col("size_delta") < 100, "3. small (< 100 B)") \
    .when(col("size_delta") < 1000, "4. medium (100 B - 1 KB)") \
    .otherwise("5. large (>= 1 KB)")

size_categories_df = edits_df \
    .withColumn("size_category", size_category) \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        window(col("event_time"), "2 minutes"),
        col("size_category")
    ) \
    .agg(
        count("*").alias("edits"),
        sum_(when(col("bot"), 1).otherwise(0)).alias("bot_edits"),
        min_("size_delta").alias("min_delta"),
        max_("size_delta").alias("max_delta")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "size_category", "edits", "bot_edits", "min_delta", "max_delta"
    )

q4 = start_console(size_categories_df, "q4_tumbling_2min_sizes", "update")

## 5. Tumbling window: activity per wiki

Useful when the producer follows several wikis, for example:

```bash
python wikistream_producer.py --wiki fr.wikipedia.org en.wikipedia.org de.wikipedia.org
```

In [12]:
per_wiki_df = edits_df \
    .withWatermark("event_time", "2 minutes") \
    .groupBy(
        window(col("event_time"), "1 minute"),
        col("server_name")
    ) \
    .agg(
        count("*").alias("edits"),
        round_(100 * avg(when(col("bot"), 1).otherwise(0)), 1).alias("bot_pct"),
        sum_("size_delta").alias("net_bytes")
    ) \
    .select(
        col("window.start").alias("window_start"),
        col("window.end").alias("window_end"),
        "server_name", "edits", "bot_pct", "net_bytes"
    )

q5 = start_console(per_wiki_df, "q5_tumbling_1min_per_wiki", "update")

## 6. Big edits alerts (stateless query)

No aggregation here: every edit that adds or removes at least 5 000 bytes is printed as soon as it arrives (`append` mode).
The same alerts are also saved in parquet in `data/big_edits` (the `data` folder of the project).

In [13]:
BIG_EDIT_BYTES = 5000

big_edits_df = edits_df \
    .filter(col("abs_delta") >= BIG_EDIT_BYTES) \
    .select("event_time", "server_name", "title", "user", "bot", "size_delta", "comment")

q6 = start_console(big_edits_df, "q6_big_edits_alerts", "append", trigger="10 seconds")

q6_parquet = big_edits_df \
    .writeStream \
    .queryName("q6_big_edits_parquet") \
    .format("parquet") \
    .option("path", "/home/jovyan/data/big_edits") \
    .option("checkpointLocation", "/home/jovyan/data/checkpoints/big_edits") \
    .outputMode("append") \
    .trigger(processingTime="1 minute") \
    .start()

## Monitoring the queries

In [14]:
# Running queries and their last progress
for q in spark.streams.active:
    progress = q.lastProgress
    rows = progress["numInputRows"] if progress else None
    rate = round(progress["processedRowsPerSecond"], 1) if progress else None
    print(f"{q.name:35} active={q.isActive}  last batch rows={rows}  rows/s={rate}")

q1_tumbling_1min_activity           active=True  last batch rows=0  rows/s=0.0
q5_tumbling_1min_per_wiki           active=True  last batch rows=368  rows/s=170.4
q2_sliding_10min_hot_pages          active=True  last batch rows=368  rows/s=85.2
q3_sliding_5min_active_users        active=True  last batch rows=368  rows/s=131.1
q0_demo_bot_1h                      active=True  last batch rows=0  rows/s=0.0
q4_tumbling_2min_sizes              active=True  last batch rows=0  rows/s=0.0
q6_big_edits_parquet                active=True  last batch rows=368  rows/s=76.3
q6_big_edits_alerts                 active=True  last batch rows=368  rows/s=322.0


In [15]:
# Read back the big edits saved in parquet (batch read)
try:
    spark.read.parquet("/home/jovyan/data/big_edits") \
        .orderBy(col("event_time").desc()) \
        .show(20, truncate=60)
except Exception as e:
    print("No parquet file yet, wait for the first micro-batch:", e)

+-------------------+----------------+-------------------------------+---------+-----+----------+------------------------------------------------------------+
|         event_time|     server_name|                          title|     user|  bot|size_delta|                                                     comment|
+-------------------+----------------+-------------------------------+---------+-----+----------+------------------------------------------------------------+
|2026-10-09 08:27:17|fr.wikipedia.org|           Guia Lopes da Laguna|    Vlaam|false|     -7349|v2.05 - [[P:HOM|Homonymies]] : Correction de 1 lien - [[B...|
|2026-10-08 15:50:56|fr.wikipedia.org|Utilisateur:Vicartbsu/Brouillon|Vicartbsu|false|     -6868|                           Résumé automatique : blanchiment.|
+-------------------+----------------+-------------------------------+---------+-----+----------+------------------------------------------------------------+



## Stop all the queries

In [16]:
for q in spark.streams.active:
    q.stop()
print("all queries stopped")

all queries stopped
